# Enveda CASMI 2026 — Google Colab GPU Data, Validation & Model Training Pipeline

**Workflow Architecture:**
1. **Mount Google Drive (`/content/drive/MyDrive/CASMI2026`)**: Persistent storage for competition Parquet datasets, candidate shards (COCONUT 2.0 + PubChem), and trained model checkpoints.
2. **Download Competition Data Directly in Colab**: Uses Kaggle's cloud-to-cloud link (`~80-200 MB/s`) to download `enveda-casmi-2026` and `rishavyadav/casmi-2026-candidate-database` straight into Google Drive once, and copies active shards to Colab's local NVMe (`/content/casmi_fast`) for fast I/O.
3. **Leak-Free `inchikey14` Cross-Validation**: Evaluates `MRR@25` across both **Class 1 (In-Library)** and **Class 2 (Out-of-Library Isomers)** using `RDKit 2026.03.3` tautomer canonicalization.
4. **Train GPU `FPNet` + `LightGBM` Multi-Spectrum Reranker**: Exports a self-contained bundle to `/content/drive/MyDrive/CASMI2026/kaggle_dataset_export` and publishes/updates your public Kaggle Dataset with a single command.

In [ ]:
# 1. Install pinned CASMI 2026 dependencies (RDKit 2026.03.3, LightGBM, Polars, Kaggle CLI)
!pip install -q rdkit==2026.03.3 lightgbm polars pyarrow kaggle tqdm

In [ ]:
# 2. Mount Google Drive & Configure Fast Local NVMe Scratch Directory
import os
import shutil
from pathlib import Path

try:
    from google.colab import drive, userdata
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

DRIVE_ROOT = Path('/content/drive/MyDrive/CASMI2026') if IN_COLAB else Path('/mnt/LMSTUDIO/CASMI')
FAST_DIR = Path('/content/casmi_fast') if IN_COLAB else (DRIVE_ROOT / 'data')
EXPORT_DIR = DRIVE_ROOT / 'kaggle_dataset_export'

for d in [DRIVE_ROOT / 'data', DRIVE_ROOT / 'candidates', FAST_DIR, EXPORT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f'Persistent Drive Root: {DRIVE_ROOT}')
print(f'Fast NVMe Scratch:     {FAST_DIR}')
print(f'Kaggle Export Bundle:  {EXPORT_DIR}')

In [ ]:
# 3. Configure Kaggle API Credentials (from Colab Secrets or ~/.kaggle/kaggle.json)
# In Colab, add KAGGLE_USERNAME and KAGGLE_KEY in the left-hand 'Secrets' (key icon) panel.
if IN_COLAB:
    try:
        os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
        os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
        print('[OK] Loaded Kaggle credentials from Colab Secrets.')
    except Exception:
        kaggle_json_drive = DRIVE_ROOT / 'kaggle.json'
        if kaggle_json_drive.exists():
            Path('/root/.kaggle').mkdir(parents=True, exist_ok=True)
            shutil.copy2(kaggle_json_drive, '/root/.kaggle/kaggle.json')
            os.chmod('/root/.kaggle/kaggle.json', 0o600)
            print('[OK] Loaded Kaggle credentials from Google Drive kaggle.json.')

In [ ]:
# 4. Download Competition Dataset & Candidate Shards once to Google Drive, then stage on NVMe
train_drive_path = DRIVE_ROOT / 'data' / 'train.parquet'
if not train_drive_path.exists():
    print('Downloading enveda-casmi-2026 to Google Drive...')
    !kaggle competitions download -c enveda-casmi-2026 -p "{DRIVE_ROOT}/data"
    !unzip -o "{DRIVE_ROOT}/data/*.zip" -d "{DRIVE_ROOT}/data" && rm -f "{DRIVE_ROOT}/data"/*.zip
else:
    print('[SKIP] train.parquet already present on Google Drive.')

coconut_dir = DRIVE_ROOT / 'candidates' / 'coconut'
if not coconut_dir.exists():
    print('Downloading CASMI 2026 Candidate Database (COCONUT 2.0 + PubChem shards)...')
    !kaggle datasets download -d rishavyadav/casmi-2026-candidate-database -p "{DRIVE_ROOT}/candidates" --unzip
else:
    print('[SKIP] Candidate database already present on Google Drive.')

# Copy active Parquet files to Colab local NVMe (/content/casmi_fast) for 10x faster reading
if IN_COLAB and train_drive_path.exists() and not (FAST_DIR / 'train.parquet').exists():
    shutil.copy2(train_drive_path, FAST_DIR / 'train.parquet')
    if (DRIVE_ROOT / 'data' / 'test.parquet').exists():
        shutil.copy2(DRIVE_ROOT / 'data' / 'test.parquet', FAST_DIR / 'test.parquet')
    print('[OK] Staged Parquet files to local NVMe.')

In [ ]:
# 5. Add `casmi26` package to Python path & Run Leak-Free Validation + GPU Training
import sys
import numpy as np
import pandas as pd

for candidate_pkg_root in [DRIVE_ROOT, Path('/mnt/LMSTUDIO/CASMI'), Path('.')]:
    if (candidate_pkg_root / 'casmi26').exists() and str(candidate_pkg_root) not in sys.path:
        sys.path.insert(0, str(candidate_pkg_root))

from casmi26.chemistry import score_mrr_at_25
from casmi26.features import clean_and_normalize_spectrum, smiles_to_fingerprint, spectrum_to_binned_vector
from casmi26.models import CASMIPipeline, SpectralLibraryIndex, SpectrumToFingerprintModel
from casmi26.retrieval import build_candidate_pool_from_sources
from casmi26.validation import assign_leak_free_folds, sample_timstof_like_validation_panel

print('casmi26 package loaded successfully.')

In [ ]:
# 6. Train Full Pipeline on GPU & Evaluate Local Leak-Free MRR@25
train_path = FAST_DIR / 'train.parquet'
if train_path.exists():
    train_df = pd.read_parquet(train_path)
    cand_shards = sorted((DRIVE_ROOT / 'candidates').rglob('*.parquet'))
else:
    from tests.test_pipeline import generate_synthetic_casmi26_data
    print('[INFO] Real train.parquet not found yet; running on synthetic verification data.')
    train_df, _ = generate_synthetic_casmi26_data()
    cand_shards = sorted((Path('/mnt/LMSTUDIO/CASMI/data/synthetic')).glob('*.parquet'))

train_df['fold'] = assign_leak_free_folds(train_df, n_splits=5, group_col='inchikey14')
tr_split = train_df[train_df['fold'] != 0].reset_index(drop=True)
va_split = train_df[train_df['fold'] == 0].reset_index(drop=True)

# Build CandidatePool (train structures + COCONUT 2.0 + PubChem shards)
candidate_pool = build_candidate_pool_from_sources(train_df=tr_split, candidate_parquet_paths=cand_shards)
print(f'Indexed {len(candidate_pool):,} unique inchikey14 candidates.')

# Prepare training tensors for GPU FPNet
max_fp_train = min(len(tr_split), 150_000)  # Increase to full 2.5M on Colab A100/L4
tr_sub = tr_split.sample(n=max_fp_train, random_state=42) if len(tr_split) > max_fp_train else tr_split

X_binned = np.zeros((len(tr_sub), 1536), dtype=np.float32)
X_nm = np.zeros(len(tr_sub), dtype=np.float32)
X_pos = np.zeros(len(tr_sub), dtype=np.float32)
Y_fp = np.zeros((len(tr_sub), 2215), dtype=np.float32)
Y_hard_neg_fp = np.zeros((len(tr_sub), 2215), dtype=np.float32)

for i, row in enumerate(tr_sub.itertuples(index=False)):
    c_mzs, c_probs = clean_and_normalize_spectrum(row.ms2_mzs, row.ms2_normalized_intensities, row.precursor_mz)
    X_binned[i] = spectrum_to_binned_vector(c_mzs, c_probs, row.precursor_mz)
    X_nm[i] = float(row.precursor_mz)
    X_pos[i] = 1.0 if str(row.ionization_mode).startswith('pos') else 0.0
    Y_fp[i] = smiles_to_fingerprint(row.normalized_smiles)
    hard_isomers = candidate_pool.sample_hard_isomer_negatives(float(row.precursor_mz), str(row.inchikey14), k_negatives=4)
    if hard_isomers:
        Y_hard_neg_fp[i] = np.mean([smiles_to_fingerprint(s) for s in hard_isomers], axis=0)

fp_model = SpectrumToFingerprintModel(use_torch_if_available=True)
fp_model.fit(X_binned, X_nm, X_pos, Y_fp, hard_neg_fps=Y_hard_neg_fp, epochs=15, batch_size=256, lr=1e-3)

# Build SpectralLibraryIndex & Evaluate on Leak-Free Validation Panel
spectral_index = SpectralLibraryIndex.build(tr_split)
val_queries, val_truth = sample_timstof_like_validation_panel(va_split, n_molecules=400)

pipeline = CASMIPipeline(
    candidate_pool=candidate_pool,
    spectral_index=spectral_index,
    fp_model=fp_model,
    ppm_tolerance=15.0,
)
pipeline.fit_reranker(val_queries, val_truth)
val_preds = pipeline.predict_top25_for_test_df(val_queries)
val_mrr, _ = score_mrr_at_25(val_preds['smiles'].tolist(), val_truth['inchikey14'].tolist(), ground_truth_is_inchikey14=True)
print(f'Leak-Free Out-of-Library Validation MRR@25: {val_mrr:.4f}')

# Rebuild full index across all folds before exporting for Kaggle Leaderboard inference
pipeline.candidate_pool = build_candidate_pool_from_sources(train_df=train_df, candidate_parquet_paths=cand_shards)
pipeline.spectral_index = SpectralLibraryIndex.build(train_df)
pipeline.save_checkpoint(EXPORT_DIR)
print(f'[SAVED] Exported Kaggle Dataset bundle to: {EXPORT_DIR}')

In [ ]:
# 7. Package & Publish/Update the Trained Model Artifact as a Kaggle Dataset
# Replace 'your-username/casmi26-trained-models' with your Kaggle username!
KAGGLE_DATASET_SLUG = 'your-username/casmi26-trained-models'

from scripts.publish_kaggle_model_dataset import prepare_dataset_metadata
prepare_dataset_metadata(EXPORT_DIR, dataset_slug=KAGGLE_DATASET_SLUG)

# Uncomment one of the commands below to push directly from Colab to Kaggle Datasets:
# !kaggle datasets create -p "{EXPORT_DIR}" --dir-mode zip --public
# !kaggle datasets version -p "{EXPORT_DIR}" -m "Colab GPU trained model update" --dir-mode zip